[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/pascal/pascal.ipynb)

# Pascal's Theorem

This example constructs the conic through five points in the projective plane and demonstrates Pascal's theorem as a sixth point moves around it. Five points in general position fix a single conic through them. Joining six points on that conic in order forms a hexagon whose three pairs of opposite sides meet in three crossings that remain collinear whether the hexagon is convex or self-crossing. Reversing the theorem reconstructs the conic itself, because the sixth points whose opposite-side crossings line up are precisely the points of the conic through the first five.

A conic is modeled as a polarity map `Plane <- Point` sending each point to its polar plane, and its curve consists of the points that lie on their own polar where `quadric(point) & point == 0`. Two lines form a degenerate quadric from the symmetrized product of their pairings, and the pencil of quadrics through four points spans the linear combinations of two such line pairs until the fifth point selects one. Joining two points with `&` gives a side line, intersecting opposite sides with `^` gives their crossing, and joining the three crossings with `&` while writing the sixth corner as `Point` produces a bilinear form `Scalar <- (Point, Point)` proportional to the quadric.

In homogeneous matrix and vector notation, a conic is represented by a symmetric matrix $C \in \mathbb{R}^{3 \times 3}$ satisfying $\mathbf{p}^{\mathsf{T}} C \mathbf{p} = 0$, the pencil through four points combines two rank-2 matrices $(\mathbf{l}_1 \mathbf{m}_1^{\mathsf{T}} + \mathbf{m}_1 \mathbf{l}_1^{\mathsf{T}})$ and $(\mathbf{l}_2 \mathbf{m}_2^{\mathsf{T}} + \mathbf{m}_2 \mathbf{l}_2^{\mathsf{T}})$ built from cross products $\mathbf{l} = \mathbf{p}_i \times \mathbf{p}_j$, and collinearity of the three opposite-side intersections requires expanding $\det[(\mathbf{p}_1 \times \mathbf{p}_2) \times (\mathbf{p}_4 \times \mathbf{p}_5),\; (\mathbf{p}_2 \times \mathbf{p}_3) \times (\mathbf{p}_5 \times \mathbf{x}),\; (\mathbf{p}_3 \times \mathbf{p}_4) \times (\mathbf{x} \times \mathbf{p}_1)] = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.math.pascal import render, scenarios
from examples.util.animation import save_figure, show_states
from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from numga.topology import Topology

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Planes are extrinsic vectors and points are intrinsic antivectors; a quadric maps each point to its polar plane.
Plane = ga.gatype.vector()                                                     # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
# Two points join into an intrinsic line, and two lines meet in an extrinsic crossing.
Line = ga.gatype.antibivector()                                                # Antibivector-
Crossing = ga.gatype.bivector()                                                # Bivector+
Quadric = ga.gatype((Plane, Point))                                            # Plane+ <- Point-
Form = ga.gatype((Scalar, Point, Point))                                       # Scalar+ <- (Point-, Point-)

hexagon_loop = Topology.loops([6])

## The quadric through five points

Two lines form a degenerate quadric whose pairing with a point multiplies the two lines' individual pairings and therefore vanishes along either line. Four points admit two such line pairs, one formed by opposite sides of their quadrilateral and one formed by its diagonals, and every quadric through those four points is a linear combination of the two pairs. Requiring the combination to vanish at the fifth point determines the quadric up to overall scale.

In [ ]:
five = scenarios.ellipse_points(scenarios.five_angles)                         # [5] Point-

In [ ]:
def quadric(points: Point) -> Quadric:
    """Five points along the trailing axis (`[..., 5] Point-`) determine the unique polarity map (`[...] Plane+ <- Point-`) vanishing on all five by combining the two line-pair quadrics through the first four so that the fifth lies on its own polar."""
    first, second, third, fourth, fifth = (points[..., index] for index in range(5))
    # Two pairs of lines pass through the first four points: opposite sides of their quadrilateral and its diagonals.
    first_lines = stack([first & second, first & third])                       # [2, ...] Line-
    second_lines = stack([third & fourth, second & fourth])                    # [2, ...] Line-
    # Symmetrizing the two lines' pairings builds a polarity map that vanishes along either line.
    line_pairs = (first_lines * (second_lines & Point) + second_lines * (first_lines & Point)) / 2   # [2, ...] Plane+ <- Point-
    side_pair, diagonal_pair = line_pairs[0], line_pairs[1]                    # [...] Plane+ <- Point-, [...] Plane+ <- Point-
    return side_pair * (diagonal_pair(fifth) & fifth) - diagonal_pair * (side_pair(fifth) & fifth)   # [...] Plane+ <- Point-


shape = quadric(five)                                                          # [] Plane+ <- Point-
print(shape.coefficient_table())

## Sweeping a sixth point and the Pascal line

Turning a non-tangent line through half a turn about the first point sweeps a sixth point once around the quadric. Along that line, `shape(point) & point` is a quadratic in the step along the line's direction with one root already at zero, so the second crossing is rational in the quadric and direction without a square root.


In [ ]:
n_frames = scenarios.n_frames
angles = scenarios.turning_angles(n_frames)
still = scenarios.still
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def hexagon_configurations(five: Point, angles: np.ndarray) -> Iterator[tuple[Point, Line, Crossing, Line]]:
    """Yields per-angle configurations `(hexagon, sides, crossings, pascal_line)` (`[6] Point-`, `[6] Line-`, `[3] Crossing+`, `[] Line-`) as a line through `five[0]` turns through `angles` and traces the sixth point once around `quadric(five)`."""
    shape = quadric(five)                                                      # [] Plane+ <- Point-
    start = five[0]                                                            # [] Point-
    for angle in angles:
        turn = (mv.xy * (-angle / 2)).exp()                                    # [] Rotor+
        # Turning a point at infinity gives the line's ideal direction point.
        heading = turn >> mv.x.dual()                                          # [] Point-
        # Along `start + t * heading`, the quadratic `2 * t * (shape(start) & heading) + t**2 * (shape(heading) & heading)`
        # already vanishes at `t = 0`, giving the second root rationally.
        sixth = (start * (shape(heading) & heading) - heading * (2 * (shape(start) & heading))).normalized()   # [] Point-
        hexagon = stack([five[index] for index in range(5)] + [sixth])         # [6] Point-
        tails, heads = hexagon_loop.incidence[1, 0] * hexagon.field()          # Point-[E], Point-[E]
        sides = (tails & heads).batch()                                        # [6] Line-
        crossings = sides[:3] ^ sides[3:]                                      # [3] Crossing+
        pascal_line = crossings[0] & crossings[2]                              # [] Line-
        yield hexagon, sides, crossings, pascal_line


states = show_states(
    hexagon_configurations(five, angles),
    render.animate,
    shape,
    name="pascal_turn",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

In [ ]:
save_figure(render.draw(shape, *states[still]), "pascal_hexagon")

The grey curve traces the quadric through the five dark corners, while the orange disc marks the moving sixth corner. Opposite sides of the hexagon share blue, green, and purple strokes, and their three red-ringed crossings glide along the red Pascal line across both convex and self-crossing configurations.

## Pascal's condition as a bilinear form

Writing the sixth corner as `Point` in the two hexagon sides that meet there turns the regressive join of the three opposite-side crossings into a bilinear form `Scalar <- (Point, Point)`. Because that form is quadratic and vanishes at all five fixed points, its ratio to `shape(probes) & probes` is constant across arbitrary points off the curve.

In [ ]:
n_probes = scenarios.n_probes
probes = mv.point(np.random.default_rng(scenarios.seed).normal(size=(n_probes, 2)))   # [n_probes] Point-

In [ ]:
def pascal_form(five: Point) -> Form:
    """Five fixed corners (`[..., 5] Point-`) define a symmetric bilinear form (`[...] Scalar+ <- (Point-, Point-)`) on the sixth corner from the regressive join of the three opposite-side crossings, which vanishes whenever those crossings are collinear."""
    first, second, third, fourth, fifth = (five[..., index] for index in range(5))
    return (((first & second) ^ (fourth & fifth))
            & ((second & third) ^ (fifth & Point))
            & ((third & fourth) ^ (Point & first)))                            # [...] Scalar+ <- (Point-, Point-)


ratios = pascal_form(five)(probes, probes) / (shape(probes) & probes)          # [n_probes] Scalar+
print("Pascal form to quadric ratio across probes:", ratios.to_array())

In [ ]:
# checks
hexagons = stack([state[0] for state in states])                               # [n_frames, 6] Point-
crossings = stack([state[2] for state in states])                              # [n_frames, 3] Crossing+
joins = crossings[:, 0] & crossings[:, 1] & crossings[:, 2]                    # [n_frames] Scalar+
sizes = crossings[:, 0].norm() * crossings[:, 1].norm() * crossings[:, 2].norm()   # [n_frames] Scalar+
assert ((shape(five) & five).abs() < 1e-11).all()
assert ((shape(hexagons) & hexagons).abs() < 1e-11).all()
assert ((joins / sizes).abs() < 1e-10).all()
assert ((ratios - ratios[0]).abs() < 1e-11 * ratios[0].abs()).all()